<a href="https://colab.research.google.com/github/md-tanjimul-islam/Ai-ml-shohoj-coding/blob/main/Assignment%20-%2005%20%3A%20Feature%20Engineering%20%26%20Data%20Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 5: Feature Engineering & Data Preparation
**Machine Learning & AI BootCamp**  
*Instructor: Fahd*  
*Difficulty Level: Easy (Unique Case Study: SoundWave Music Platform)*

---

### 📌 Instructions & Guidelines:
1. Complete all 8 exercises by writing your code directly below each `# TODO:` comment.
2. The assignment uses `soundwave_music_users.csv`, a real-world dataset from a music streaming service tracking listening behavior, skips, subscription tiers, and churn risk.
3. Follow the steps sequentially. Each exercise tests concepts from **Class 9 (Feature Engineering & Encoding)** and **Class 10 (Train-Test Split & Data Leakage Prevention)**.
4. Always follow the **Golden Rule of Data Leakage Prevention**: fit your transformers strictly on training data!


In [ ]:
from google.colab import files
uploaded = files.upload()


Saving soundwave_music_users.csv to soundwave_music_users.csv


---
## Student Information
- **Student Name:** *Md. Tanjimul Islam*
- **Batch:** *MLB-2*
- **Date:** *29-06-2026*


In [ ]:
# Run this cell first to import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

print("All required libraries successfully imported!")


All required libraries successfully imported!


---
## Exercise 1: Loading & Inspecting the SoundWave Dataset
1. Load `soundwave_music_users.csv` into a DataFrame named `df`.
2. Display the shape of the DataFrame and the first 5 rows.
3. Check the class distribution of the target column `churn_risk` using `.value_counts()`.


In [ ]:
df = pd.read_csv('soundwave_music_users.csv')
print(df.shape)
print(df.head())
print(df['churn_risk'].value_counts())

(40, 10)
   user_id signup_date  age subscription_tier primary_genre  \
0  SW-1001  2023-01-15   24              Free       Hip-Hop   
1  SW-1002  2022-11-04   38              Gold          Rock   
2  SW-1003  2023-03-20   29            Silver           Pop   
3  SW-1004  2021-08-12   45              Gold          Jazz   
4  SW-1005  2023-05-28   19              Free    Electronic   

   daily_listen_mins  songs_skipped_per_day  playlists_created    device_type  \
0                 45                     18                  1         Mobile   
1                130                      4                  8        Desktop   
2                 85                      9                  4         Mobile   
3                150                      3                 12  Smart Speaker   
4                 35                     22                  0         Mobile   

   churn_risk  
0           1  
1           0  
2           0  
3           0  
4           1  
churn_risk
0    30
1    10
Na

---
## Exercise 2: Datetime Feature Engineering
Timestamps contain rich seasonal and tenure signals.
1. Convert `signup_date` into a `datetime64[ns]` data type using `pd.to_datetime()`.
2. Create a new column `signup_month` containing the integer month (1 to 12).
3. Create a new column `signup_day_name` containing the day of the week name (e.g. Monday, Tuesday) using `.dt.day_name()`.
4. Calculate `tenure_days`: the number of days between the user's signup date and `'2024-01-01'` (hint: `(pd.to_datetime('2024-01-01') - df['signup_date']).dt.days`).
5. Display the first 5 rows of these newly created datetime features.


In [ ]:
df['signup_date'] = pd.to_datetime(df['signup_date'])
df['signup_month'] = df['signup_date'].dt.month
df['signup_day_name'] = df['signup_date'].dt.day_name()
df['tenure_days'] = (pd.to_datetime('2024-01-01') - df['signup_date']).dt.days
print(df[['signup_date', 'signup_month', 'signup_day_name', 'tenure_days']].head())

  signup_date  signup_month signup_day_name  tenure_days
0  2023-01-15             1          Sunday          351
1  2022-11-04            11          Friday          423
2  2023-03-20             3          Monday          287
3  2021-08-12             8        Thursday          872
4  2023-05-28             5          Sunday          218


---
## Exercise 3: Domain-Specific Feature Creation
In music streaming, a user who skips a lot of songs relative to their listening time might be unsatisfied and at higher risk of churning!
1. Calculate the estimated number of songs listened to per day, assuming an average song length of **3.5 minutes**:  
   $$\text{songs\_listened} = \frac{\text{daily\_listen\_mins}}{3.5}$$
2. Create a new feature `skip_rate`:  
   $$\text{skip\_rate} = \frac{\text{songs\_skipped\_per\_day}}{\text{songs\_listened}}$$
3. Create a binary flag `is_curator`: $1$ if `playlists_created >= 5`, and $0$ otherwise.
4. Display the first 5 rows of `['daily_listen_mins', 'songs_skipped_per_day', 'skip_rate', 'is_curator']`.


In [ ]:
songs_listened = df['daily_listen_mins'] / 3.5
df['skip_rate'] = df['songs_skipped_per_day'] / songs_listened
df['is_curator'] = (df['playlists_created'] >= 5).astype(int)
print(df[['daily_listen_mins', 'songs_skipped_per_day', 'skip_rate', 'is_curator']].head())

   daily_listen_mins  songs_skipped_per_day  skip_rate  is_curator
0                 45                     18   1.400000           0
1                130                      4   0.107692           1
2                 85                      9   0.370588           0
3                150                      3   0.070000           1
4                 35                     22   2.200000           0


---
## Exercise 4: Categorical Encoding
1. **Ordinal Encoding**: `subscription_tier` has an inherent hierarchy:  
   `Free` (0) < `Bronze` (1) < `Silver` (2) < `Gold` (3).  
   Map `subscription_tier` into a new column `tier_code` using a Python dictionary and `.map()`.
2. **One-Hot Encoding**: Encode `device_type` using `pd.get_dummies()` with `drop_first=True` and `dtype=int`.
3. Display the first 5 rows of the encoded columns.


In [ ]:
tier_map = {'Free': 0, 'Bronze': 1, 'Silver': 2, 'Gold': 3}
df['tier_code'] = df['subscription_tier'].map(tier_map)
device_encoded = pd.get_dummies(df['device_type'], drop_first=True, dtype=int)
print(df[['subscription_tier', 'tier_code']].head())
print(device_encoded.head())

  subscription_tier  tier_code
0              Free          0
1              Gold          3
2            Silver          2
3              Gold          3
4              Free          0
   Mobile  Smart Speaker
0       1              0
1       0              0
2       1              0
3       0              1
4       1              0


---
## Exercise 5: Discretization & Binning
1. Using `pd.cut()`, bin user `age` into 3 meaningful generational segments:
   - **Bins**: `[18, 25, 40, 70]`
   - **Labels**: `['Young (18-25)', 'Adult (26-40)', 'Mature (41+)']`
2. Store the result in a column named `age_group`.
3. Print the number of users in each age group using `.value_counts()`.


In [ ]:
df['age_group'] = pd.cut(df['age'], bins=[18, 25, 40, 70], labels=['Young (18-25)', 'Adult (26-40)', 'Mature (41+)'], include_lowest=True)
print(df['age_group'].value_counts())

age_group
Adult (26-40)    19
Mature (41+)     13
Young (18-25)     8
Name: count, dtype: int64


---
## Exercise 6: Stratified Train-Test Split
1. Define feature matrix `X` and target `y`:
   - Let `y = df['churn_risk']`.
   - Let `X` be the numerical features: `['daily_listen_mins', 'songs_skipped_per_day', 'tenure_days', 'skip_rate', 'tier_code']`.
2. Perform an **80/20 train-test split** using `train_test_split()`.
   - Set `random_state=42`.
   - **Crucial**: Use `stratify=y` to preserve the exact class ratio of churn risk!
3. Print the percentage of churn risk (`y == 1`) in both `y_train` and `y_test` to verify that stratification worked perfectly.


In [ ]:
y = df['churn_risk']
X = df[['daily_listen_mins', 'songs_skipped_per_day', 'tenure_days', 'skip_rate', 'tier_code']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print((y_train == 1).mean() * 100)
print((y_test == 1).mean() * 100)

25.0
25.0


---
## Exercise 7: Leakage-Free Feature Scaling
1. Initialize `StandardScaler()`.
2. **Strict Leakage Prevention**:
   - Fit and transform the scaler **strictly on `X_train`**: `X_train_scaled = scaler.fit_transform(X_train)`.
   - Transform `X_test` using the parameters learned from train: `X_test_scaled = scaler.transform(X_test)`.
3. Verify that the mean of each column in `X_train_scaled` is approximately `0` and standard deviation is approximately `1`.


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(np.round(X_train_scaled.mean(axis=0), 2))
print(np.round(X_train_scaled.std(axis=0), 2))

[-0.  0.  0. -0. -0.]
[1. 1. 1. 1. 1.]


---
## Exercise 8: Scikit-Learn Preprocessing Pipeline (Synthesis)
Let's bring everything together into a clean, leakage-free Scikit-Learn Pipeline!
1. From the original DataFrame `df`, select:
   - Numerical features: `['daily_listen_mins', 'songs_skipped_per_day', 'tenure_days']`
   - Categorical features: `['primary_genre', 'device_type']`
2. Build a `ColumnTransformer` named `preprocessor`:
   - Scale numerical features with `StandardScaler()`.
   - Encode categorical features with `OneHotEncoder(drop='first', sparse_output=False)`.
3. Build a complete `Pipeline` named `model_pipeline`:
   - Step 1: `('preprocessor', preprocessor)`
   - Step 2: `('classifier', LogisticRegression(random_state=42))`
4. Split the data using `train_test_split(..., test_size=0.20, random_state=42, stratify=df['churn_risk'])`.
5. Fit `model_pipeline` on the training set and print the test accuracy score!


In [ ]:
num_features = ['daily_listen_mins', 'songs_skipped_per_day', 'tenure_days']
cat_features = ['primary_genre', 'device_type']
preprocessor = ColumnTransformer(transformers=[('num', StandardScaler(), num_features), ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_features)])
model_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', LogisticRegression(random_state=42))])
X = df[num_features + cat_features]
y = df['churn_risk']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=df['churn_risk'])
model_pipeline.fit(X_train, y_train)
y_pred = model_pipeline.predict(X_test)
print(accuracy_score(y_test, y_pred))

1.0


---
### 🎉 Congratulations on Completing Assignment 5!
You have mastered:
- Engineering temporal and domain-specific behavioral features.
- Correctly applying ordinal and one-hot encoding.
- Splitting data with stratification to handle class imbalance.
- Enforcing strict train-test separation to eliminate data leakage.
- Building end-to-end preprocessing pipelines with Scikit-Learn!
